# Sistema de Enriquecimiento Semántico y Desambiguación de Perfiles Profesionales

Este notebook demuestra el sistema de enriquecimiento de perfiles profesionales que:
- Desambigua términos polisemánticos (Java, automatización, React, etc.)
- Infiere el sector profesional principal
- Extrae habilidades explícitas e implícitas
- Analiza competencias profesionales
- Normaliza términos y crea mapas de sinónimos
- Infiere el nivel de seniority

## Arquitectura del Sistema

El sistema utiliza LangGraph para orquestar un pipeline de análisis semántico:

```
START → analyze_context → infer_sector → disambiguate_terms → 
extract_skills → analyze_competencies → normalize_terms → 
infer_seniority → assemble_enriched_profile → END
```

### Nodos del Grafo:

1. **analyze_context**: Analiza el contexto del perfil extrayendo indicadores clave
2. **infer_sector**: Infiere el sector profesional principal con scoring de confianza
3. **disambiguate_terms**: Resuelve términos ambiguos usando contexto sectorial
4. **extract_skills**: Extrae habilidades explícitas e implícitas
5. **analyze_competencies**: Identifica competencias blandas y de liderazgo
6. **normalize_terms**: Crea mapas de sinónimos y términos canónicos
7. **infer_seniority**: Determina el nivel de seniority del candidato
8. **assemble_enriched_profile**: Ensambla el perfil enriquecido final

In [ ]:
# Imports
import json
from datetime import datetime
from pprint import pprint

from profile_enrichment.enrichment_agent import enrich_profile, enrichment_agent

## Caso de Prueba 1: Técnico de Automatización Industrial

Este caso demuestra la desambiguación de términos como:
- **automatización**: ¿Software o industrial?
- **Python**: ¿Lenguaje de programación?
- **SCADA/PLC**: Tecnologías específicas de automatización industrial

In [ ]:
# Perfil de ejemplo: Técnico de Automatización Industrial
industrial_profile = {
    "id": "uuid-auto2025",
    "full_name": "Luis Hidalgo",
    "heading": "Técnico de automatización y sistemas de control industrial",
    "description": (
        "Especialista en automatización de plantas desalinizadoras mediante SCADA, "
        "PLC (Siemens S7-300/1200), HMI y sistemas de telecontrol de procesos. "
        "Experiencia en integración de sensores y actuadores para optimización energética, "
        "validación de alarmas y control remoto. En proyectos recientes, también colaboré "
        "con equipos de desarrollo en pruebas automáticas con Python y selenium para la "
        "validación de software industrial."
    ),
    "experience_years": 6,
    "skills": [
        {"name": "automatización", "level": "Advanced"},
        {"name": "SCADA", "level": "Advanced"},
        {"name": "PLC", "level": "Advanced"},
        {"name": "Python", "level": "Intermediate"},
        {"name": "pruebas automáticas", "level": "Intermediate"},
        {"name": "telecontrol", "level": "Intermediate"},
    ],
    "experiences": [
        {
            "role": "Técnico de Automatización",
            "description": (
                "Diseño y mantenimiento de sistemas de control PLC y SCADA en plantas "
                "desalinizadoras, integración de sensores y HMI para supervisión remota."
            ),
            "company": "ICR Agua y Energía",
        },
        {
            "role": "Ingeniero de Validación",
            "description": (
                "Colaboración con desarrolladores en automatización de pruebas con "
                "Python y Selenium sobre entorno SCADA industrial."
            ),
            "company": "Tecnología Industrial Avanzada",
        },
    ],
}

candidate_id = "ec142b18-befd-4f89-9f3d-98a7b53c3fr2"
metadata = {
    "timestamp": datetime.now().isoformat(),
    "source": "linkedin_scraper_v2",
    "sector_objetivo": "industrial|utilities",
}

In [ ]:
# Ejecutar el enriquecimiento
print("🔄 Enriqueciendo perfil industrial...\n")
enriched_industrial = enrich_profile(
    raw_profile=industrial_profile,
    candidate_id=candidate_id,
    metadata=metadata,
)

print("✅ Enriquecimiento completado\n")

### Resultados: Insights Clave

In [ ]:
# Mostrar insights principales
key_insights = enriched_industrial["semantic_enrichment"]["key_insights"]
print("📊 INSIGHTS CLAVE\n" + "="*50)
print(f"Sector Principal: {key_insights['primary_sector']}")
print(f"Confianza Sector: {key_insights['sector_confidence']:.2f}")
print(f"Nivel Seniority: {key_insights['seniority_level']}")
print(f"Años Experiencia: {key_insights['years_experience']}")
print(f"Skills Identificados: {key_insights['total_skills_identified']}")
print(f"Competencias: {key_insights['competencies_count']}")
print(f"Términos Desambiguados: {key_insights['disambiguated_terms_count']}")

### Resultados: Desambiguación de Términos

In [ ]:
# Mostrar términos desambiguados
disambiguation_map = enriched_industrial["semantic_enrichment"]["disambiguation_map"]
print("\n🔍 DESAMBIGUACIÓN DE TÉRMINOS\n" + "="*50)
for term, details in disambiguation_map.items():
    print(f"\n📌 Término: '{term}'")
    print(f"   Significado: {details['meaning']}")
    print(f"   Dominio: {details['domain']}")
    print(f"   Confianza: {details['confidence']:.2f}")

### Resultados: Skills Explícitos e Implícitos

In [ ]:
# Mostrar skills
skills = enriched_industrial["semantic_enrichment"]["structured_skills"]
print("\n🛠️ SKILLS EXPLÍCITOS\n" + "="*50)
for skill in skills["explicit"][:5]:  # Primeros 5
    print(f"\n• {skill['name']} ({skill['category']})")
    print(f"  Nivel: {skill['level']}")
    print(f"  Evidencia: {skill['evidence'][:100]}...")

print("\n\n💡 SKILLS IMPLÍCITOS\n" + "="*50)
for skill in skills["implicit"][:5]:  # Primeros 5
    print(f"\n• {skill['name']} ({skill['category']})")
    print(f"  Nivel: {skill['level']}")
    print(f"  Confianza: {skill['confidence']:.2f}")
    print(f"  Evidencia: {skill['evidence'][:100]}...")

### Resultados: Competencias Profesionales

In [ ]:
# Mostrar competencias
competencies = enriched_industrial["semantic_enrichment"]["structured_competencies"]
print("\n🎯 COMPETENCIAS PROFESIONALES\n" + "="*50)
for comp in competencies[:5]:  # Primeras 5
    print(f"\n• {comp['competency']} ({comp['type']})")
    print(f"  Nivel: {comp['level']}")
    print(f"  Fuente: {comp['source']}")
    print(f"  Evidencia: {comp['evidence'][:100]}...")

### Resultados: Normalización y Sinónimos

In [ ]:
# Mostrar normalización
synonym_map = enriched_industrial["semantic_enrichment"]["synonym_map"]
print("\n📚 MAPAS DE SINÓNIMOS\n" + "="*50)
for term, synonyms in list(synonym_map.items())[:5]:  # Primeros 5
    print(f"\n• {term}:")
    print(f"  Variaciones: {', '.join(synonyms)}")

### Resultados: Razonamiento y Evidencia (Explainability)

In [ ]:
# Mostrar explainability
explainability = enriched_industrial["explainability"]
print("\n📖 EXPLICABILIDAD DEL ANÁLISIS\n" + "="*50)
print("\n🔬 Razonamiento Sector:")
print(explainability["sector_reasoning"])
print("\n📋 Evidencia Sector:")
for i, evidence in enumerate(explainability["sector_evidence"][:3], 1):
    print(f"{i}. {evidence}")

print("\n\n🎓 Razonamiento Seniority:")
print(explainability["seniority_reasoning"])
print("\n📋 Evidencia Seniority:")
for i, evidence in enumerate(explainability["seniority_evidence"][:3], 1):
    print(f"{i}. {evidence}")

### Exportar Perfil Enriquecido

In [ ]:
# Guardar perfil enriquecido
output_path = "enriched_profile_industrial.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(enriched_industrial, f, indent=2, ensure_ascii=False)

print(f"\n💾 Perfil enriquecido guardado en: {output_path}")

## Caso de Prueba 2: Desarrollador Full-stack

Este caso demuestra la desambiguación en un contexto de desarrollo de software:
- **React**: Framework JavaScript
- **Python**: Lenguaje de programación
- **automatización**: Test automation (no industrial)

In [ ]:
# Perfil de ejemplo: Desarrollador Full-stack
software_profile = {
    "id": "uuid-dev2025",
    "full_name": "Ana Martínez",
    "heading": "Full-stack Developer especializada en React y Node.js",
    "description": (
        "Desarrolladora con experiencia en arquitectura de aplicaciones web modernas. "
        "Especializada en React para frontend y Node.js con Express para backend. "
        "Implementación de APIs RESTful, integración con bases de datos PostgreSQL y MongoDB. "
        "Experiencia en automatización de testing con Jest y Cypress. Lideré un equipo de "
        "5 desarrolladores para crear una plataforma e-commerce completa con React y Node.js."
    ),
    "experience_years": 4,
    "skills": [
        {"name": "React", "level": "Advanced"},
        {"name": "Node.js", "level": "Advanced"},
        {"name": "JavaScript", "level": "Advanced"},
        {"name": "TypeScript", "level": "Intermediate"},
        {"name": "Python", "level": "Intermediate"},
        {"name": "automatización", "level": "Intermediate"},
        {"name": "PostgreSQL", "level": "Intermediate"},
    ],
    "experiences": [
        {
            "role": "Senior Full-stack Developer",
            "description": (
                "Lideré el desarrollo de plataforma e-commerce con React, Node.js, y PostgreSQL. "
                "Arquitectura de microservicios con Docker y Kubernetes. Implementación de CI/CD con GitHub Actions."
            ),
            "company": "TechStartup Inc",
        },
        {
            "role": "Frontend Developer",
            "description": (
                "Desarrollo de componentes React reutilizables, gestión de estado con Redux, "
                "automatización de tests con Jest y Cypress."
            ),
            "company": "Digital Agency",
        },
    ],
}

software_candidate_id = "ec142b18-befd-4f89-9f3d-98a7b53c3fr3"
software_metadata = {
    "timestamp": datetime.now().isoformat(),
    "source": "linkedin_scraper_v2",
}

In [ ]:
# Ejecutar el enriquecimiento
print("🔄 Enriqueciendo perfil software...\n")
enriched_software = enrich_profile(
    raw_profile=software_profile,
    candidate_id=software_candidate_id,
    metadata=software_metadata,
)

print("✅ Enriquecimiento completado\n")

In [ ]:
# Comparación de desambiguación entre perfiles
print("\n🔀 COMPARACIÓN DE DESAMBIGUACIÓN\n" + "="*70)
print("\nTérmino: 'automatización'\n")

industrial_auto = enriched_industrial["semantic_enrichment"]["disambiguation_map"].get("automatización", {})
software_auto = enriched_software["semantic_enrichment"]["disambiguation_map"].get("automatización", {})

print("PERFIL INDUSTRIAL:")
if industrial_auto:
    print(f"  Significado: {industrial_auto.get('meaning', 'N/A')}")
    print(f"  Dominio: {industrial_auto.get('domain', 'N/A')}")

print("\nPERFIL SOFTWARE:")
if software_auto:
    print(f"  Significado: {software_auto.get('meaning', 'N/A')}")
    print(f"  Dominio: {software_auto.get('domain', 'N/A')}")

print("\n" + "="*70)
print("🎯 El sistema correctamente distingue que 'automatización' significa")
print("   CONTROL INDUSTRIAL en un contexto vs TEST AUTOMATION en el otro.")

## Visualización del Grafo

El sistema está construido como un grafo LangGraph con nodos secuenciales de análisis.

In [ ]:
# Intentar visualizar el grafo (requiere graphviz)
try:
    from IPython.display import Image, display
    display(Image(enrichment_agent.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"No se pudo visualizar el grafo: {e}")
    print("\nEstructura del grafo:")
    print("START → analyze_context → infer_sector → disambiguate_terms →")
    print("extract_skills → analyze_competencies → normalize_terms →")
    print("infer_seniority → assemble_enriched_profile → END")

## Resumen del Sistema

### Capacidades Principales:

1. **Desambiguación Robusta**: Resuelve términos polisemánticos usando contexto sectorial
2. **Inferencia de Sector**: Determina el sector profesional con alta confianza
3. **Extracción de Skills**: Identifica habilidades explícitas e implícitas
4. **Análisis de Competencias**: Detecta soft skills y capacidades de liderazgo
5. **Normalización**: Crea mapas de sinónimos para mejor matching
6. **Inferencia de Seniority**: Evalúa el nivel profesional del candidato
7. **Explainability**: Proporciona razonamiento y evidencia para cada inferencia

### Output Estructurado:

El perfil enriquecido incluye:
- **Datos originales preservados**: Todo el perfil original se mantiene
- **Semantic Enrichment**: Análisis completo y estructurado
- **Key Insights**: Resumen ejecutivo de hallazgos principales
- **Structured Data**: Datos estructurados listos para consumo
- **Explainability**: Razonamiento y evidencia para transparencia

### Uso en Downstream Systems:

El output JSON puede ser consumido por:
- Sistemas de matching candidato-oferta
- Motores de búsqueda y recomendación
- Dashboards de análisis de talento
- APIs de enriquecimiento de datos
- Sistemas de scoring y ranking

## Caso de Prueba 3: Perfil Híbrido (Industrial Software)

Este caso demuestra un perfil híbrido donde el candidato trabaja en el cruce entre industrial y software.

In [ ]:
# Perfil híbrido: Industrial IoT Developer
hybrid_profile = {
    "id": "uuid-hybrid2025",
    "full_name": "Carlos Rodríguez",
    "heading": "Industrial IoT Developer",
    "description": (
        "Desarrollador especializado en soluciones IoT para entornos industriales. "
        "Experiencia integrando sensores industriales con plataformas cloud usando Python, "
        "Node.js y React para dashboards de monitoreo. Trabajo con protocolos MQTT, OPC-UA, "
        "y Modbus para conectar PLCs y sistemas SCADA a soluciones de analítica en tiempo real. "
        "Implementación de edge computing para procesamiento local en plantas industriales."
    ),
    "experience_years": 5,
    "skills": [
        {"name": "Python", "level": "Advanced"},
        {"name": "React", "level": "Advanced"},
        {"name": "Node.js", "level": "Intermediate"},
        {"name": "MQTT", "level": "Advanced"},
        {"name": "OPC-UA", "level": "Advanced"},
        {"name": "PLC", "level": "Intermediate"},
        {"name": "SCADA", "level": "Intermediate"},
        {"name": "IoT", "level": "Advanced"},
    ],
    "experiences": [
        {
            "role": "IoT Solutions Architect",
            "description": (
                "Diseño de arquitecturas IoT para manufactura inteligente. Integración de "
                "sistemas SCADA legacy con plataformas cloud (AWS IoT, Azure IoT). Desarrollo "
                "de dashboards React para visualización de datos industriales en tiempo real."
            ),
            "company": "Industrial IoT Solutions",
        },
        {
            "role": "Industrial Software Developer",
            "description": (
                "Desarrollo de software para integración de PLCs con sistemas de analítica. "
                "Python para procesamiento de datos, Node.js para APIs, comunicación con "
                "protocolos industriales (Modbus, OPC-UA)."
            ),
            "company": "Manufacturing Tech Corp",
        },
    ],
}

hybrid_candidate_id = "ec142b18-befd-4f89-9f3d-98a7b53c3fr4"
hybrid_metadata = {
    "timestamp": datetime.now().isoformat(),
    "source": "linkedin_scraper_v2",
}

In [ ]:
# Ejecutar el enriquecimiento
print("🔄 Enriqueciendo perfil híbrido...\n")
enriched_hybrid = enrich_profile(
    raw_profile=hybrid_profile,
    candidate_id=hybrid_candidate_id,
    metadata=hybrid_metadata,
)

print("✅ Enriquecimiento completado\n")

# Mostrar insights
hybrid_insights = enriched_hybrid["semantic_enrichment"]["key_insights"]
print("📊 INSIGHTS CLAVE - PERFIL HÍBRIDO\n" + "="*50)
print(f"Sector Principal: {hybrid_insights['primary_sector']}")
print(f"Confianza Sector: {hybrid_insights['sector_confidence']:.2f}")
print(f"Nivel Seniority: {hybrid_insights['seniority_level']}")
print(f"\n🔬 Razonamiento:")
print(enriched_hybrid["explainability"]["sector_reasoning"])

## Conclusión

El sistema de enriquecimiento semántico es capaz de:

✅ Desambiguar términos polisemánticos según contexto sectorial
✅ Inferir sectores profesionales con alta confianza
✅ Identificar skills explícitos e implícitos
✅ Analizar competencias blandas y de liderazgo
✅ Normalizar términos para mejor matching
✅ Determinar niveles de seniority
✅ Proporcionar explainability completa

El output JSON enriquecido está listo para ser consumido por sistemas downstream
de matching, búsqueda, y análisis de talento.